# 00 — Kiểm tra môi trường và dữ liệu thô cho bộ Yoochoose

Notebook này **chỉ kiểm tra tính sẵn sàng**, không cài đặt package mới và không sửa dữ liệu gốc.

Các hạng mục kiểm tra được thiết kế riêng cho đặc thù của **Yoochoose (RecSys Challenge 2015)**:

1. Kiểm tra Python kernel (`D:\DL\.venv`) và khả năng import package nội bộ `btl_k03`.
2. Kiểm tra các thư viện thực sự dùng trong pipeline (đặc biệt là **`pyarrow`** để đọc nhanh 33 triệu dòng dưới giới hạn RAM và **`torch`** cho giai đoạn huấn luyện; *không cần RecBole* vì 3 mô hình GRU4Rec/NARM/SASRec đã được cài đặt chuẩn PyTorch trong `src/btl_k03/models.py`).
3. Kiểm tra phần cứng: GPU/CUDA, dung lượng VRAM, RAM hệ thống và dung lượng trống ổ đĩa `D:\`.
4. Kiểm tra 2 file dữ liệu thô `yoochoose-clicks.dat` (~1,42 GB) và `yoochoose-buys.dat` (~53 MB): định dạng cột, dòng đầu/cuối, số dòng và SHA-256.
5. Tạo và xác minh các thư mục đầu ra tách biệt hoàn toàn cho Yoochoose (`data/interim/yoochoose`, `data/processed/yoochoose`, `runs/yoochoose`, `results/yoochoose`) để **không ảnh hưởng đến kết quả Diginetica đã hoàn thành**.

In [1]:
from __future__ import annotations

import ctypes
import hashlib
import importlib
import importlib.util
import os
import platform
import shutil
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "BTL-K03-tong-quan-de-tai.md").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc DA-3.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from btl_k03.paths import get_dataset_paths, ensure_output_dirs

YOO_PATHS = get_dataset_paths("yoochoose")
DIGI_PATHS = get_dataset_paths("diginetica")

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_CLICKS_FILE : {YOO_PATHS['raw_clicks_file']}")
print(f"RAW_BUYS_FILE   : {YOO_PATHS['raw_buys_file']}")
print(f"PROCESSED_YOO   : {YOO_PATHS['processed_dir']}")
print(f"RESULTS_YOO     : {YOO_PATHS['results_dir']}")

PROJECT_ROOT    : D:\DA-3
RAW_CLICKS_FILE : D:\DA-3\data\raw\yoochoose-clicks.dat\yoochoose-clicks.dat
RAW_BUYS_FILE   : D:\DA-3\data\raw\yoochoose-buys.dat\yoochoose-buys.dat
PROCESSED_YOO   : D:\DA-3\data\processed\yoochoose
RESULTS_YOO     : D:\DA-3\results\yoochoose


## 1. Kiểm tra Python kernel

Dự án đang sử dụng môi trường ảo `D:\DL\.venv` (Python 3.12). Nếu `sys.executable` trỏ sang Python khác, hãy chọn lại kernel `dl-venv` trước khi chạy EDA/tiền xử lý.

In [2]:
EXPECTED_PYTHON = Path(r"D:\DL\.venv\Scripts\python.exe").resolve()
CURRENT_PYTHON = Path(sys.executable).resolve()
USING_EXPECTED_VENV = CURRENT_PYTHON == EXPECTED_PYTHON

print(f"Python version      : {platform.python_version()}")
print(f"Python executable   : {CURRENT_PYTHON}")
print(f"Expected executable : {EXPECTED_PYTHON}")
print(f"Đúng venv           : {USING_EXPECTED_VENV}")
print(f"Operating system    : {platform.platform()}")

if not USING_EXPECTED_VENV:
    print("\nCẢNH BÁO: Notebook chưa chạy trên D:\\DL\\.venv. Hãy đổi kernel sang 'DL Venv' (dl-venv).")

Python version      : 3.12.6
Python executable   : D:\DL\.venv\Scripts\python.exe
Expected executable : D:\DL\.venv\Scripts\python.exe
Đúng venv           : True
Operating system    : Windows-11-10.0.26200-SP0


## 2. Kiểm tra thư viện cần thiết

Không cần cài thêm thư viện ngoài môi trường đã dùng cho Diginetica. Lưu ý **`pyarrow`** là bắt buộc để đọc `yoochoose-clicks.dat` (33 triệu dòng) trong ~2–3 giây mà vẫn giữ RAM ổn định.

In [3]:
DEPENDENCIES = [
    ("numpy", "numpy", "CORE", "Tính toán mảng vector hóa"),
    ("pandas", "pandas", "CORE", "Thao tác bảng dữ liệu"),
    ("pyarrow", "pyarrow", "CORE", "Đọc nhanh 33M dòng CSV & ghi Parquet zstd"),
    ("scipy", "scipy", "ANALYSIS", "Spearman & kiểm định thống kê"),
    ("scikit-learn", "sklearn", "ANALYSIS", "PCA, t-SNE & Logistic Regression"),
    ("matplotlib", "matplotlib", "VIZ", "Vẽ biểu đồ"),
    ("seaborn", "seaborn", "VIZ", "Vẽ biểu đồ thống kê"),
    ("PyYAML", "yaml", "TOOLING", "Đọc/ghi cấu hình YAML"),
    ("tqdm", "tqdm", "TOOLING", "Thanh tiến trình"),
    ("nbformat", "nbformat", "TOOLING", "Xử lý notebook"),
    ("ipykernel", "ipykernel", "TOOLING", "Jupyter kernel"),
    ("pytest", "pytest", "TOOLING", "Kiểm thử tự động"),
    ("torch", "torch", "TRAINING", "Huấn luyện GRU4Rec / NARM / SASRec"),
]


def get_pkg_ver(dist: str, mod) -> str:
    try:
        return package_version(dist)
    except PackageNotFoundError:
        return str(getattr(mod, "__version__", "unknown"))


dep_rows = []
for dist, mod_name, stage, purpose in DEPENDENCIES:
    found = importlib.util.find_spec(mod_name) is not None
    mod, err = None, None
    if found:
        try:
            mod = importlib.import_module(mod_name)
        except Exception as exc:
            err = f"{type(exc).__name__}: {exc}"
    ok = found and err is None
    dep_rows.append(
        {
            "package": dist,
            "stage": stage,
            "ok": ok,
            "version": get_pkg_ver(dist, mod) if ok else "-",
            "purpose": purpose,
            "error": err,
        }
    )

print(f"{'PACKAGE':<16} {'STAGE':<10} {'STATUS':<9} {'VERSION':<18} PURPOSE")
print("-" * 86)
for r in dep_rows:
    st = "OK" if r["ok"] else "MISSING"
    print(f"{r['package']:<16} {r['stage']:<10} {st:<9} {r['version']:<18} {r['purpose']}")
    if r["error"]:
        print(f"  Import error: {r['error']}")

PACKAGE          STAGE      STATUS    VERSION            PURPOSE
--------------------------------------------------------------------------------------
numpy            CORE       OK        2.5.2              Tính toán mảng vector hóa
pandas           CORE       OK        2.3.3              Thao tác bảng dữ liệu
pyarrow          CORE       OK        24.0.0             Đọc nhanh 33M dòng CSV & ghi Parquet zstd
scipy            ANALYSIS   OK        1.18.1             Spearman & kiểm định thống kê
scikit-learn     ANALYSIS   OK        1.9.0              PCA, t-SNE & Logistic Regression
matplotlib       VIZ        OK        3.11.1             Vẽ biểu đồ
seaborn          VIZ        OK        0.13.2             Vẽ biểu đồ thống kê
PyYAML           TOOLING    OK        6.0.3              Đọc/ghi cấu hình YAML
tqdm             TOOLING    OK        4.69.0             Thanh tiến trình
nbformat         TOOLING    OK        5.11.1             Xử lý notebook
ipykernel        TOOLING    OK        7.

## 3. Kiểm tra tài nguyên phần cứng (RAM, ổ đĩa, GPU/CUDA)

Với file raw `1,42 GB` (33 triệu dòng), việc nắm rõ RAM khả dụng và VRAM GPU giúp chọn quy mô lát cắt Yoochoose hợp lý (ví dụ `1/32` hoặc `1/64`) để cả 3 bước tiền xử lý, huấn luyện 3 mô hình × 2 seed và phân tích hình học đều chạy ổn định.

In [4]:
class MEMORYSTATUSEX(ctypes.Structure):
    _fields_ = [
        ("dwLength", ctypes.c_ulong),
        ("dwMemoryLoad", ctypes.c_ulong),
        ("ullTotalPhys", ctypes.c_ulonglong),
        ("ullAvailPhys", ctypes.c_ulonglong),
        ("ullTotalPageFile", ctypes.c_ulonglong),
        ("ullAvailPageFile", ctypes.c_ulonglong),
        ("ullTotalVirtual", ctypes.c_ulonglong),
        ("ullAvailVirtual", ctypes.c_ulonglong),
        ("ullAvailExtendedVirtual", ctypes.c_ulonglong),
    ]


if os.name == "nt":
    mem = MEMORYSTATUSEX()
    mem.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
    ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(mem))
    total_ram_gb = mem.ullTotalPhys / (1024**3)
    avail_ram_gb = mem.ullAvailPhys / (1024**3)
    print(f"RAM tổng / khả dụng : {total_ram_gb:.2f} GB / {avail_ram_gb:.2f} GB")

disk = shutil.disk_usage(PROJECT_ROOT)
print(f"CPU logical cores   : {os.cpu_count()}")
print(f"Ổ đĩa chứa dự án    : còn trống {disk.free / (1024**3):.2f} GB / {disk.total / (1024**3):.2f} GB")

torch_ok = next(r["ok"] for r in dep_rows if r["package"] == "torch")
if torch_ok:
    import torch

    print(f"PyTorch version     : {torch.__version__}")
    print(f"CUDA available      : {torch.cuda.is_available()} (build: {torch.version.cuda})")
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(
            f"  GPU {i}: {prop.name} | VRAM={prop.total_memory / (1024**3):.2f} GB | "
            f"SM={prop.major}.{prop.minor}"
        )

RAM tổng / khả dụng : 15.69 GB / 2.83 GB
CPU logical cores   : 12
Ổ đĩa chứa dự án    : còn trống 5.80 GB / 104.64 GB
PyTorch version     : 2.14.0+cu130
CUDA available      : True (build: 13.0)
  GPU 0: NVIDIA GeForce MX550 | VRAM=2.00 GB | SM=7.5


## 4. Kiểm tra dữ liệu thô Yoochoose (`clicks` & `buys`)

Hai file Yoochoose là CSV **không có header**:
- `yoochoose-clicks.dat`: 4 cột (`session_id, timestamp, item_id, category`)
- `yoochoose-buys.dat`: 5 cột (`session_id, timestamp, item_id, price, quantity`)

In [5]:
def inspect_raw_file(path: Path, expected_cols: int, compute_hash: bool = True) -> dict:
    exists = path.is_file()
    if not exists:
        return {"file": path.name, "exists": False}

    size_bytes = path.stat().st_size
    with path.open("rb") as f:
        head = [f.readline().decode("utf-8", errors="replace").strip() for _ in range(3)]
        f.seek(max(0, size_bytes - 4096))
        tail = f.read().decode("utf-8", errors="replace").strip().splitlines()[-3:]

    digest = hashlib.sha256() if compute_hash else None
    line_count = 0
    with path.open("rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            line_count += chunk.count(b"\n")
            if digest is not None:
                digest.update(chunk)

    col_counts_ok = all(len(line.split(",")) == expected_cols for line in head + tail if line)
    return {
        "file": path.name,
        "exists": True,
        "size_mb": round(size_bytes / (1024**2), 2),
        "lines": line_count,
        "expected_cols": expected_cols,
        "format_ok": col_counts_ok,
        "sha256": digest.hexdigest().upper() if digest is not None else "-",
        "sample_head": head[0] if head else "",
        "sample_tail": tail[-1] if tail else "",
    }


raw_checks = [
    inspect_raw_file(YOO_PATHS["raw_clicks_file"], expected_cols=4, compute_hash=True),
    inspect_raw_file(YOO_PATHS["raw_buys_file"], expected_cols=5, compute_hash=True),
]

for item in raw_checks:
    print("=" * 80)
    print(f"File       : {item['file']}")
    print(f"Tồn tại    : {item['exists']}")
    if item["exists"]:
        print(f"Dung lượng : {item['size_mb']:,.2f} MB | Số dòng: {item['lines']:,}")
        print(f"Đúng {item['expected_cols']} cột : {item['format_ok']}")
        print(f"SHA-256    : {item['sha256']}")
        print(f"Dòng đầu   : {item['sample_head']}")
        print(f"Dòng cuối  : {item['sample_tail']}")

File       : yoochoose-clicks.dat
Tồn tại    : True
Dung lượng : 1,417.92 MB | Số dòng: 33,003,944
Đúng 4 cột : True
SHA-256    : 0DEAF041A4F60955928FE8430E525A1DBF38E9013C7482420973E4AA492FA762
Dòng đầu   : 1,2014-04-07T10:51:09.277Z,214536502,0
Dòng cuối  : 11299811,2014-09-24T19:02:25.146Z,214848658,S
File       : yoochoose-buys.dat
Tồn tại    : True
Dung lượng : 53.01 MB | Số dòng: 1,150,753
Đúng 5 cột : True
SHA-256    : 0C5C0F5084DA2FF703C15D7D72298136E4243806ADC3E29ECCFA920CE3E24E23
Dòng đầu   : 420374,2014-04-06T18:44:58.314Z,214537888,12462,1
Dòng cuối  : 11423202,2014-09-26T18:49:34.026Z,214560500,5549,1


## 5. Khởi tạo thư mục đầu ra riêng cho Yoochoose & kiểm tra cô lập với Diginetica

In [6]:
ensure_output_dirs("yoochoose")
(PROJECT_ROOT / "configs" / "yoochoose").mkdir(parents=True, exist_ok=True)

check_dirs = {
    "configs_yoo": YOO_PATHS["config_dir"],
    "interim_yoo": YOO_PATHS["interim_dir"],
    "processed_yoo": YOO_PATHS["processed_dir"],
    "runs_yoo": YOO_PATHS["runs_dir"],
    "results_yoo": YOO_PATHS["results_dir"],
    "tables_yoo": YOO_PATHS["tables_dir"],
    "figures_yoo": YOO_PATHS["figures_dir"],
}

print(f"{'KEY':<16} {'EXISTS':<8} {'WRITABLE':<10} PATH")
print("-" * 88)
for name, d in check_dirs.items():
    print(f"{name:<16} {str(d.is_dir()):<8} {str(os.access(d, os.W_OK)):<10} {d}")

digi_ok = (DIGI_PATHS["processed_dir"] / "dataset_statistics.json").is_file()
config_ok = (YOO_PATHS["config_dir"] / "data.yaml").is_file()
print(f"\nDữ liệu Diginetica cũ vẫn nguyên vẹn : {digi_ok}")
print(f"Cấu hình configs/yoochoose/data.yaml : {config_ok}")

KEY              EXISTS   WRITABLE   PATH
----------------------------------------------------------------------------------------
configs_yoo      True     True       D:\DA-3\configs\yoochoose
interim_yoo      True     True       D:\DA-3\data\interim\yoochoose
processed_yoo    True     True       D:\DA-3\data\processed\yoochoose
runs_yoo         True     True       D:\DA-3\runs\yoochoose
results_yoo      True     True       D:\DA-3\results\yoochoose
tables_yoo       True     True       D:\DA-3\results\yoochoose\tables
figures_yoo      True     True       D:\DA-3\results\yoochoose\figures

Dữ liệu Diginetica cũ vẫn nguyên vẹn : True
Cấu hình configs/yoochoose/data.yaml : True


## 6. Tổng kết sẵn sàng

In [7]:
missing_pkgs = [r["package"] for r in dep_rows if not r["ok"]]
raw_ok = all(r.get("exists") and r.get("format_ok") for r in raw_checks)
ready_all = USING_EXPECTED_VENV and (not missing_pkgs) and raw_ok

print("TỔNG KẾT MÔI TRƯỜNG YOOCHOOSE")
print("=" * 60)
print(f"Đúng venv D:\\DL\\.venv  : {USING_EXPECTED_VENV}")
print(f"Thư viện bị thiếu       : {missing_pkgs or 'Không'}")
print(f"Dữ liệu raw hợp lệ      : {raw_ok}")
print(f"Sẵn sàng chạy 01 & 02   : {ready_all}")

TỔNG KẾT MÔI TRƯỜNG YOOCHOOSE
Đúng venv D:\DL\.venv  : True
Thư viện bị thiếu       : Không
Dữ liệu raw hợp lệ      : True
Sẵn sàng chạy 01 & 02   : True
